# ZAIC 2026 LifeMate — v9: synthetic do LLM lớn viết tay cho 11 app chỉ có ở public

**Khác v7:** thay 440 spec sinh bằng code (`gen_*.jsonl`, rập khuôn) bằng **285 spec viết tay** (`synth_specs_v9/llm_*.jsonl`):
lời nói tự nhiên hơn, có từ địa phương, tiếng Anh lẫn, sửa lời giữa câu, điều kiện đúng/sai, bẫy 2 đối tượng gần giống,
history tái dùng ID không đọc lại, và đủ các enum khó (`reason` trả hàng, `direction` cảnh báo, `type` tàu/xe, `payment`, mã ngôn ngữ).
Tất cả đã được replay trên simulator của BTC: 285/285 OK (lint của `synth_build`).

Mọi thứ khác **giữ nguyên v7/v4**: adapter train trên `train_fit` + synthetic, `--drop-empty` cho synthetic, không `--detach-prefix`, không ASR-noisy SFT.

Mốc so sánh trên val (ASR thật `asr_ft_v4`): v4 **35.06 / 54.11**, v7 34.88 / 56.02. Public: v4 44.46 (Docker 45.29), v7 44.72. GPU **A100**, khoảng 2–3 giờ.

`SPEC_GLOBS` có thể thêm `synth_specs/gen_*.jsonl` nếu muốn thử gộp cả bộ cũ (v7 chứng minh bộ cũ hoà v4).


In [ ]:
# ==== CẤU HÌNH ====
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BATCH      = 32
LLM_EPOCHS = 1
TRAIN_ALL  = False    # False: train trên train_fit, giữ val để đo (so sánh sạch với v4/v7)
                      # True : train trên cả 1545 mẫu — chỉ bật cho lần train CUỐI, khi v9 đã chứng minh tốt hơn trên public
USE_ASR    = False    # giữ False (bài học v5)
DROP_EMPTY = True     # bỏ mẫu synthetic có đáp án [] (bài học v6); đặt False để thử giữ 46 mẫu [] viết tay
DATA_URL   = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR  = "/content/drive/MyDrive/ZAIC"
ASR_PROD   = DRIVE_DIR + "/asr_ft_v4"          # đổi sang thư mục ASR large-v3 nếu v8 thắng
WORK, DATA_RAW, RES = "/content/work", "/content/data_raw", "/content/results_v9"
SPEC_GLOBS = [WORK + "/synth_specs_v9/llm_*.jsonl"]   # nằm sẵn trong lifemate_baseline.zip; thêm WORK + "/synth_specs/gen_*.jsonl" để gộp bộ cũ
LLM_OUT    = "/content/ft_v9"


In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU"
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0); print(GPU_NAME, DTYPE)
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob, subprocess, json, time
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(os.path.join(DRIVE_DIR, "lifemate_baseline.zip")).extractall(WORK)
specs = sorted(p for g in SPEC_GLOBS for p in glob.glob(g))
assert specs, "Chưa có file spec nào khớp " + str(SPEC_GLOBS) + " — lifemate_baseline.zip trên Drive chưa phải bản có synth_specs_v9"
print("synth spec files:", len(specs), [os.path.basename(p) for p in specs])
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        assert subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL]).returncode == 0, "Tải dữ liệu lỗi"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
IDS  = "" if TRAIN_ALL else "--ids splits/train_fit.txt"
ASRF = "--asr asr_train_oof.json" if USE_ASR else ""
if USE_ASR:
    shutil.copy(os.path.join(DRIVE_DIR, "asr_train_oof.json"), WORK + "/asr_train_oof.json")
print("DATA:", DATA)

In [ ]:
%%capture pipout
!pip install -q vllm peft soundfile
!pip uninstall -y torchao

In [ ]:
print(pipout.stderr[-800:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|peft) " | tee {RES}/versions.txt
%cd {WORK}
!cat splits/val_unseen.txt splits/val_seen.txt > splits/val_all.txt

## 1. Dựng và kiểm tra dữ liệu synthetic bằng simulator của BTC

In [ ]:
%cd {WORK}
!python -m lifemate.synth_build --root {DATA} --tools-from public_test --specs {' '.join(SPEC_GLOBS)} 2>&1 | tee {RES}/synth_build.log | tail -25

In [ ]:
import collections as C
rows = [json.loads(l) for l in open(f"{DATA}/synth/synth.jsonl", encoding="utf-8")]
tag = lambda r: "oos" if not r["label"]["tool_calls"] else ("ask" if any(c["name"].startswith("sys.") for c in r["label"]["tool_calls"]) else "act")
apps = C.Counter(a for r in rows for a in r["context"]["installed_apps"])
n_synth = len(rows)
print("mẫu synthetic dựng được:", n_synth, "|", dict(C.Counter(tag(r) for r in rows)))
print("app:", apps.most_common(14))
assert n_synth >= 250, "quá ít mẫu qua được kiểm tra — xem synth_build.log rồi sửa spec"
open(f"{RES}/synth_stats.txt", "w").write(f"{n_synth} mẫu\n{dict(C.Counter(tag(r) for r in rows))}\n{apps.most_common()}\n")

## 2. Dữ liệu SFT (công thức v4: bản gốc + bản đổi tên app)

In [ ]:
%cd {WORK}
!python -m lifemate.sft_data --root {DATA} {IDS} --aug-copies 1 {ASRF} --out sft_real.jsonl
!python -m lifemate.sft_data --root {DATA} --split synth --aug-copies 1 {"--drop-empty" if DROP_EMPTY else ""} --out sft_synth.jsonl
with open("sft_all.jsonl", "w", encoding="utf-8") as out:
    for p in ["sft_real.jsonl", "sft_synth.jsonl"]:
        out.writelines(open(p, encoding="utf-8"))
n = sum(1 for _ in open("sft_all.jsonl")); print("tổng ví dụ SFT:", n)

## 3. Train (full gradient như v4 — **không** dùng `--detach-prefix`). A100 khoảng 2–3 giờ

In [ ]:
%cd {WORK}
!python -m lifemate.train_lora --data sft_all.jsonl --model {BASE_MODEL} --out {LLM_OUT} --epochs {LLM_EPOCHS} 2>&1 | grep -vi warn
assert os.path.exists(f"{LLM_OUT}/merged/config.json"), "train lỗi: xem log phía trên"
shutil.copy(f"{LLM_OUT}/train_log.txt", f"{RES}/train_log.txt")
shutil.copytree(f"{LLM_OUT}/adapter", os.path.join(DRIVE_DIR, "ft_v9_adapter"), dirs_exist_ok=True)

## 4. Đánh giá trên val với ASR thật (v4: 35.06 / 54.11 — mốc cần vượt)

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split train --ids splits/val_all.txt --model {ASR_PROD} --dtype {DTYPE} --out asr_val.json 2>&1 | grep -E "^model=|^WER"
sc = {}
for S in ["val_unseen", "val_seen"]:
    !python -m lifemate.run --root {DATA} --split train --ids splits/{S}.txt --backend vllm --model {LLM_OUT}/merged --dtype {DTYPE} --batch {BATCH} --transcripts asr_val.json --out runs/v9_{S} 2>&1 | tail -1
    !python scripts/score.py --root {DATA} --split train --pred runs/v9_{S}/submission.jsonl --by-tag > {RES}/score_v9_{S}.txt
    !python -m lifemate.report --root {DATA} --split train --pred runs/v9_{S}/submission.jsonl --show 6 > {RES}/report_v9_{S}.txt
    sc[S] = float(open(f"{RES}/score_v9_{S}.txt").readline().split()[-1])
    !head -9 {RES}/report_v9_{S}.txt
print("v9:", sc, "| v4: {'val_unseen': 35.06, 'val_seen': 54.11} | v7: {'val_unseen': 34.88, 'val_seen': 56.02}")
if TRAIN_ALL:
    print("CHÚ Ý: TRAIN_ALL=True nên val đã nằm trong tập train — điểm này cao giả tạo, chỉ nhìn điểm public")
json.dump(sc, open(f"{RES}/e2e_v9.json", "w"))

## 5. Public → `submission_public_v9.jsonl` (mốc cần vượt: **44.72** của v7 / 44.46 của v4)

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split public_test --model {ASR_PROD} --dtype {DTYPE} --out asr_public.json 2>&1 | tail -1
!python -m lifemate.run --root {DATA} --split public_test --backend vllm --model {LLM_OUT}/merged --dtype {DTYPE} --batch {BATCH} --transcripts asr_public.json --out runs/public_v9 2>&1 | tail -1
shutil.copy("runs/public_v9/submission.jsonl", f"{RES}/submission_public_v9.jsonl")
shutil.copy("runs/public_v9/submission.jsonl", os.path.join(DRIVE_DIR, "submission_public_v9.jsonl"))
shutil.copy("runs/public_v9/traces.jsonl", f"{RES}/traces_public_v9.jsonl")
shutil.copy("asr_public.json", f"{RES}/asr_public.json")
sub = [json.loads(l) for l in open("runs/public_v9/submission.jsonl")]
ask = sum(1 for x in sub if any(c["name"].startswith("sys.") for c in x["tool_calls"]))
empty = sum(1 for x in sub if not x["tool_calls"])
print(f"dòng {len(sub)} | ask_missing_info {ask} (v4: 92, v6: 43, v7: xem results_v7) | rỗng {empty} (v4: 64, v6: 106)")

In [ ]:
%cd {DATA}/public_test
!python -m zaic_tools --split public_test --root .. --validate {RES}/submission_public_v9.jsonl 2>&1 | tail -3 | tee {RES}/validate_public_v9.txt

## 6. Đóng gói

In [ ]:
open(f"{RES}/config.txt", "w").write(f"BASE={BASE_MODEL}\nTRAIN_ALL={TRAIN_ALL}\nUSE_ASR={USE_ASR}\nDROP_EMPTY={DROP_EMPTY}\nSPECS={SPEC_GLOBS}\nSFT={n}\nsynth={n_synth}\nGPU={GPU_NAME}\n")
z = shutil.make_archive("/content/results_v9", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, "results_v9.zip")); print("saved results_v9.zip")
from google.colab import files
files.download(z); files.download(f"{RES}/submission_public_v9.jsonl")